In [2]:
import numpy as np
import pandas as pd

# ==============================================================================
# 1. HÀM MISSING_REPORT() NGẮN GỌN
# ==============================================================================
def missing_report(df):
    """Hàm thống kê số lượng và tỷ lệ % dữ liệu khuyết thiếu của các cột."""
    miss_count = df.isnull().sum()
    miss_pct = (miss_count / len(df)) * 100
    report = pd.DataFrame(
        {"missing_count": miss_count, "missing_percentage (%)": miss_pct}
    )
    # Chỉ trả về các cột có dữ liệu thiếu và sắp xếp giảm dần
    return (
        report[report["missing_count"] > 0]
        .sort_values(by="missing_count", ascending=False)
        .round(2)
    )


# Load dataset gốc
df = pd.read_csv("salary_survey_raw.csv")

print("--- BÁO CÁO TRƯỚC KHI XỬ LÝ MISSING ---")
print(missing_report(df))
print(f"Tổng số hàng ban đầu: {len(df)}\n")


# ==============================================================================
# 2. SỬA ĐỔI DTYPE (DATA TYPES) CHO TOÀN BỘ DATASET
# ==============================================================================
print("--- BƯỚC 1: CHUẨN HÓA KIỂU DỮ LIỆU (DTYPE) ---")

# 2.1 SỬA LỖI TẠI ĐÂY: Thêm format="mixed" để tránh làm mất 1.359 dòng timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", format="mixed")

# 2.2 TỐI ƯU THÊM: Chuẩn hóa cột tiền tệ trước để phục vụ việc nhóm lương chính xác hơn
df["currency"] = df["currency"].astype(str).str.strip().str.upper()

# 2.3 Ép kiểu dữ liệu số cho 'annual_salary' (Xóa bỏ ký tự lạ $, dấu phẩy, khoảng trắng)
df["annual_salary"] = (
    df["annual_salary"]
    .astype(str)
    .str.replace(r"[$,\s]", "", regex=True)
)
df["annual_salary"] = pd.to_numeric(df["annual_salary"], errors="coerce")

# 2.4 Ép kiểu dữ liệu số cho 'additional_monetary_comp'
df["additional_monetary_comp"] = (
    df["additional_monetary_comp"]
    .astype(str)
    .str.replace(r"[$,\s]", "", regex=True)
)
df["additional_monetary_comp"] = pd.to_numeric(
    df["additional_monetary_comp"], errors="coerce"
)

print("Kiểu dữ liệu sau khi ép kiểu thành công:")
print(df[["timestamp", "annual_salary", "additional_monetary_comp", "currency"]].dtypes)
print("\n")


# ==============================================================================
# 3. XỬ LÝ TRÙNG LẶP (DUPLICATES)
# ==============================================================================
print("--- BƯỚC 2: XỬ LÝ DỮ LIỆU TRÙNG LẶP (DUPLICATES) ---")
duplicate_count = df.duplicated().sum()
print(f"Số lượng dòng trùng lặp hoàn toàn phát hiện: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("-> Đã xóa bỏ các dòng trùng lặp.")
print("\n")


# ==============================================================================
# 4. HOÀN THÀNH XỬ LÝ MISSING VALUES THEO LOGIC NGHIỆP VỤ
# ==============================================================================
print(
    "--- BƯỚC 3: XỬ LÝ MISSING VALUES (Giải quyết câu hỏi bài học) ---"
)

# --- CÂU HỎI 1: TẠI SAO KHÔNG FILLNA(MEAN) CHO LƯƠNG? ---
mean_salary = df["annual_salary"].mean()
median_salary = df["annual_salary"].median()
print(f"[Minh chứng Lương] Mean: {mean_salary:.2f} | Median: {median_salary:.2f}")
print("-> Giải thích bằng code: Mean lớn hơn hẳn Median chứng tỏ dữ liệu bị lệch phải nặng do Outlier.")
print("--- CẢI TIẾN: Điền khuyết bằng MEDIAN theo cặp nhóm [Industry, Currency] để không lệch tỷ giá ---\n")

# Thực hiện điền khuyết lương bằng median của từng Industry + Currency, nếu nhóm đó không tồn tại median thì lấy global median
df["annual_salary"] = df.groupby(["industry", "currency"])["annual_salary"].transform(
    lambda x: x.fillna(x.median() if not x.dropna().empty else median_salary)
)
# Đảm bảo fill nốt nếu có nhóm cá biệt hoàn toàn trống
df["annual_salary"] = df["annual_salary"].fillna(median_salary)


# --- CÂU HỎI 2: XỬ LÝ CỘT 'country' (Null = 8%) ---
df["country"] = df["country"].str.strip().str.title()

country_mapping = {
    "Us": "United States",
    "Usa": "United States",
    "U.S.": "United States",
    "United States Of America": "United States",
    "Uk": "United Kingdom",
    "U.K.": "United Kingdom",
}
df["country"] = df["country"].replace(country_mapping)

def impute_country(row):
    if pd.isnull(row["country"]):
        if pd.notnull(row["us_state"]) or row["city"] in [
            "Boston",
            "Los Angeles",
            "New York",
        ]:
            return "United States"
        else:
            return "Unknown"
    return row["country"]

df["country"] = df.apply(impute_country, axis=1)
print("-> Đã chuẩn hóa text và sửa đổi cột 'country' dựa trên logic phân bố vùng miền.\n")


# --- CÂU HỎI 3: XỬ LÝ CỘT 'years_of_experience' (Giữ tối đa thông tin) ---
df["years_of_experience_in_field"] = df["years_of_experience_in_field"].fillna("Unknown")
df["years_of_experience_overall"] = df["years_of_experience_overall"].fillna("Unknown")
print("-> Đã xử lý 'years_of_experience' bằng cách gán nhóm 'Unknown' (Giữ lại 100% số hàng).\n")


# --- BỔ SUNG: XỬ LÝ TOÀN BỘ CÁC CỘT NULL CÒN LẠI TRONG DATASET ---
df["additional_monetary_comp"] = df["additional_monetary_comp"].fillna(0)
df["additional_context_on_job_title"] = df["additional_context_on_job_title"].fillna("No Context")
df["income_context"] = df["income_context"].fillna("No Context")
df["us_state"] = df["us_state"].fillna("Not Applicable")
df["city"] = df["city"].fillna("Unknown")
df["gender"] = df["gender"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")


# ==============================================================================
# KẾT QUẢ CUỐI CÙNG
# ==============================================================================
print("--- BÁO CÁO SAU KHI XỬ LÝ TOÀN BỘ MISSING ---")
final_report = missing_report(df)
if final_report.empty:
    print("Chúc mừng! Không còn cột nào bị khuyết thiếu (Missing = 0).")
else:
    print(final_report)

print(f"\nTổng số hàng sau khi làm sạch: {len(df)}")

--- BÁO CÁO TRƯỚC KHI XỬ LÝ MISSING ---
                                 missing_count  missing_percentage (%)
income_context                            2269                   81.04
us_state                                  1813                   64.75
city                                      1562                   55.79
additional_monetary_comp                  1538                   54.93
additional_context_on_job_title            969                   34.61
race                                       766                   27.36
annual_salary                              391                   13.96
country                                    115                    4.11
gender                                      69                    2.46
Tổng số hàng ban đầu: 2800

--- BƯỚC 1: CHUẨN HÓA KIỂU DỮ LIỆU (DTYPE) ---
Kiểu dữ liệu sau khi ép kiểu thành công:
timestamp                   datetime64[us]
annual_salary                      float64
additional_monetary_comp           float64
curre